# Integrating MCP into LangGraph Workflows

**LangGraph** gives you an explicit state machine for agent workflows — nodes, edges, checkpoints, and human interrupts. **MCP (Model Context Protocol)** gives you a standard way to discover tools and resources from remote servers instead of hard-coding a local registry.

Together they solve a common production problem:

```
┌─────────────────────────────────────────────────────────┐
│  LangGraph workflow (in your app)                       │
│  ┌──────────┐   ┌─────────┐   ┌──────────┐            │
│  │ prefetch │ → │  model  │ ⇄ │ ToolNode │            │
│  │ resources│   │  node   │   │ (MCP)    │            │
│  └────┬─────┘   └─────────┘   └────┬─────┘            │
│       │                             │                  │
│       │         MCP Client          │                  │
└───────┼─────────────────────────────┼──────────────────┘
        │         JSON-RPC            │
        ▼                             ▼
   resources/read              tools/call
        │                             │
        └──────── ShopCo MCP Server ──┘
```

This notebook wires **ShopCo Support Hub** end to end:

1. An in-process **MCP server** exposes order tools and policy resources
2. An **MCP client** discovers capabilities at runtime
3. A **bridge** converts MCP tool manifests into LangChain `StructuredTool` objects for `ToolNode`
4. A **LangGraph** graph prefetches resources, runs an offline planner (or live LLM), and executes MCP-backed tools
5. Optional **checkpointing** and a **human gate** before destructive MCP tools

Everything is self-contained. Set `USE_LIVE_LLM = True` only when you want a real model to drive tool selection.

In [ ]:
"""
ShopCo Support — LangGraph + MCP integration lab.
"""

import json
import os
import re
import uuid
from dataclasses import dataclass, field
from datetime import datetime, timezone
from typing import Annotated, Any, Callable, Literal

from langchain_core.messages import AIMessage, HumanMessage, SystemMessage, ToolMessage
from langchain_core.tools import StructuredTool
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition
from pydantic import BaseModel, Field, create_model
from typing_extensions import TypedDict

OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "sk-proj-placeholder-replace-with-your-real-key")
os.environ.setdefault("OPENAI_API_KEY", OPENAI_API_KEY)

USE_LIVE_LLM = False


def pretty(obj: Any) -> str:
    return json.dumps(obj, indent=2, default=str)


# --- ShopCo backing store ---

ORDERS: dict[str, dict[str, Any]] = {
    "ORD-1001": {
        "order_id": "ORD-1001",
        "customer": "alice@example.com",
        "status": "shipped",
        "total_usd": 89.50,
        "items": ["Wireless Mouse", "USB-C Hub"],
        "tracking": "1Z999AA10123456784",
    },
    "ORD-1002": {
        "order_id": "ORD-1002",
        "customer": "bob@example.com",
        "status": "processing",
        "total_usd": 42.00,
        "items": ["Desk Lamp"],
        "tracking": None,
    },
}

POLICIES: dict[str, str] = {
    "returns": "# Returns\n30-day window. Unused items only. Refunds in 5-7 business days.",
    "shipping": "# Shipping\nFree standard shipping over $50. Tracking emailed at carrier scan.",
}

REFUND_LOG: list[dict[str, Any]] = []

print(f"ShopCo corpus: {len(ORDERS)} orders, {len(POLICIES)} policies")

---

## 1. Why Combine MCP with LangGraph?

| Layer | Responsibility |
|-------|----------------|
| **MCP server** | Owns tools/resources, enforces schemas, hides CRM/DB details |
| **MCP client** | Discovery, `tools/call`, `resources/read` |
| **LangGraph** | Orchestration — when to prefetch, when to call model, when to execute tools, HITL interrupts |
| **ToolNode** | Executes LangChain tools that *delegate* to MCP |

Without MCP, every new integration means editing Python tool registries inside your graph. With MCP, the graph stays stable while servers add or version tools independently.

**Key integration rule:** MCP is the **wire**; LangGraph is the **control plane**. Do not put business orchestration inside the MCP server.

---

## 2. Minimal MCP Server (ShopCo)

We reuse the same JSON-RPC surface as a production MCP server, implemented in-process for teaching clarity.

In [ ]:
@dataclass
class JsonRpcRequest:
    method: str
    params: dict[str, Any] = field(default_factory=dict)
    id: str = field(default_factory=lambda: str(uuid.uuid4()))


@dataclass
class JsonRpcResponse:
    id: str
    result: Any = None
    error: dict[str, Any] | None = None

    def to_dict(self) -> dict[str, Any]:
        payload: dict[str, Any] = {"jsonrpc": "2.0", "id": self.id}
        if self.error is not None:
            payload["error"] = self.error
        else:
            payload["result"] = self.result
        return payload


SHOPCO_TOOLS: list[dict[str, Any]] = [
    {
        "name": "get_order",
        "description": "Fetch live order details by order ID. Do not use for static policy text.",
        "inputSchema": {
            "type": "object",
            "properties": {"order_id": {"type": "string", "pattern": "^ORD-[0-9]{4}$"}},
            "required": ["order_id"],
            "additionalProperties": False,
        },
        "annotations": {"readOnlyHint": True},
    },
    {
        "name": "create_refund_request",
        "description": "Open a refund ticket. Destructive — may require human approval.",
        "inputSchema": {
            "type": "object",
            "properties": {
                "order_id": {"type": "string"},
                "reason": {"type": "string", "enum": ["damaged", "wrong_item", "late_delivery", "changed_mind"]},
                "amount_usd": {"type": "number", "minimum": 0.01},
            },
            "required": ["order_id", "reason", "amount_usd"],
            "additionalProperties": False,
        },
        "annotations": {"destructiveHint": True},
    },
]

SHOPCO_RESOURCES: list[dict[str, Any]] = [
    {"uri": f"shopco://policies/{k}", "name": f"{k} policy", "mimeType": "text/markdown"}
    for k in POLICIES
]


def _tool_get_order(args: dict[str, Any]) -> dict[str, Any]:
    order = ORDERS.get(args["order_id"])
    if not order:
        return {"isError": True, "content": [{"type": "text", "text": f"Order {args['order_id']} not found"}]}
    return {"content": [{"type": "text", "text": pretty(order)}]}


def _tool_create_refund(args: dict[str, Any]) -> dict[str, Any]:
    ticket_id = f"RF-{uuid.uuid4().hex[:8].upper()}"
    record = {
        "ticket_id": ticket_id,
        "order_id": args["order_id"],
        "reason": args["reason"],
        "amount_usd": args["amount_usd"],
        "status": "pending_human_review" if args["amount_usd"] > 25 else "auto_approved",
        "created_at": datetime.now(timezone.utc).isoformat(),
    }
    REFUND_LOG.append(record)
    return {"content": [{"type": "text", "text": pretty(record)}]}


TOOL_HANDLERS: dict[str, Callable[[dict[str, Any]], dict[str, Any]]] = {
    "get_order": _tool_get_order,
    "create_refund_request": _tool_create_refund,
}


class ShopCoMCPServer:
    def __init__(self) -> None:
        self._ready = False

    def handle(self, req: JsonRpcRequest) -> JsonRpcResponse:
        if req.method == "initialize":
            self._ready = True
            return JsonRpcResponse(id=req.id, result={"protocolVersion": "2024-11-05", "serverInfo": {"name": "shopco-mcp"}})
        if not self._ready:
            return JsonRpcResponse(id=req.id, error={"code": -32002, "message": "not initialized"})
        if req.method == "tools/list":
            return JsonRpcResponse(id=req.id, result={"tools": SHOPCO_TOOLS})
        if req.method == "tools/call":
            name = req.params["name"]
            args = req.params.get("arguments", {})
            handler = TOOL_HANDLERS.get(name)
            if not handler:
                return JsonRpcResponse(id=req.id, error={"code": -32601, "message": f"unknown tool {name}"})
            return JsonRpcResponse(id=req.id, result=handler(args))
        if req.method == "resources/list":
            return JsonRpcResponse(id=req.id, result={"resources": SHOPCO_RESOURCES})
        if req.method == "resources/read":
            uri = req.params["uri"]
            key = uri.split("/")[-1]
            if key not in POLICIES:
                return JsonRpcResponse(id=req.id, error={"code": -32000, "message": "unknown resource"})
            return JsonRpcResponse(id=req.id, result={"contents": [{"uri": uri, "mimeType": "text/markdown", "text": POLICIES[key]}]})
        return JsonRpcResponse(id=req.id, error={"code": -32601, "message": f"unknown method {req.method}"})


class MCPClient:
    def __init__(self, server: ShopCoMCPServer) -> None:
        self.server = server
        self.tools: list[dict[str, Any]] = []
        self.resources: list[dict[str, Any]] = []

    def connect(self) -> None:
        self._rpc("initialize", {})
        self.tools = self._rpc("tools/list", {})["tools"]
        self.resources = self._rpc("resources/list", {})["resources"]

    def _rpc(self, method: str, params: dict[str, Any]) -> Any:
        resp = self.server.handle(JsonRpcRequest(method=method, params=params)).to_dict()
        if "error" in resp:
            raise RuntimeError(resp["error"])
        return resp["result"]

    def call_tool(self, name: str, arguments: dict[str, Any]) -> dict[str, Any]:
        return self._rpc("tools/call", {"name": name, "arguments": arguments})

    def read_resource(self, uri: str) -> str:
        return self._rpc("resources/read", {"uri": uri})["contents"][0]["text"]


mcp_client = MCPClient(ShopCoMCPServer())
mcp_client.connect()
print("MCP discovered:", [t["name"] for t in mcp_client.tools])

---

## 3. MCP → LangChain Tool Bridge

`ToolNode` expects LangChain `BaseTool` instances. The bridge:

1. Reads MCP `inputSchema` from discovery
2. Builds a Pydantic `args_schema` dynamically
3. Wraps `client.call_tool()` as the tool `func`

This is the same role as `langchain-mcp-adapters` in production — shown here explicitly so you see every layer.

In [ ]:
PY_TYPE_MAP = {"string": str, "number": float, "integer": int, "boolean": bool}


def json_schema_to_pydantic(tool_name: str, schema: dict[str, Any]) -> type[BaseModel]:
    props = schema.get("properties", {})
    required = set(schema.get("required", []))
    fields: dict[str, Any] = {}
    for key, spec in props.items():
        py_type = PY_TYPE_MAP.get(spec.get("type", "string"), str)
        if key in required:
            fields[key] = (py_type, Field(description=spec.get("description", "")))
        else:
            fields[key] = (py_type | None, Field(default=None, description=spec.get("description", "")))
    return create_model(f"{tool_name}_Input", **fields)


def mcp_result_to_text(result: dict[str, Any]) -> str:
    content = result.get("content", [])
    if not content:
        return pretty(result)
    text = content[0].get("text", str(content[0]))
    return f"ERROR: {text}" if result.get("isError") else text


class MCPToolBridge:
    """Converts discovered MCP tools into LangChain StructuredTools."""

    def __init__(self, client: MCPClient) -> None:
        self.client = client
        self._manifest_index = {t["name"]: t for t in client.tools}
        self.call_log: list[dict[str, Any]] = []

    def call(self, name: str, arguments: dict[str, Any]) -> str:
        started = datetime.now(timezone.utc)
        result = self.client.call_tool(name, arguments)
        self.call_log.append({
            "tool": name,
            "arguments": arguments,
            "is_error": bool(result.get("isError")),
            "ts": started.isoformat(),
        })
        return mcp_result_to_text(result)

    def as_langchain_tools(self) -> list[StructuredTool]:
        tools: list[StructuredTool] = []
        for manifest in self.client.tools:
            name = manifest["name"]
            args_model = json_schema_to_pydantic(name, manifest["inputSchema"])
            bridge = self

            def _invoke(args_model=args_model, name=name, **kwargs: Any) -> str:
                validated = args_model(**kwargs)
                payload = validated.model_dump(exclude_none=True)
                return bridge.call(name, payload)

            tools.append(StructuredTool(
                name=name,
                description=manifest["description"],
                func=_invoke,
                args_schema=args_model,
            ))
        return tools

    def is_destructive(self, tool_name: str) -> bool:
        manifest = self._manifest_index.get(tool_name, {})
        return bool(manifest.get("annotations", {}).get("destructiveHint"))


bridge = MCPToolBridge(mcp_client)
LANGCHAIN_TOOLS = bridge.as_langchain_tools()
print("LangChain tools from MCP:", [t.name for t in LANGCHAIN_TOOLS])

---

## 4. Graph State — Messages + MCP Context

LangGraph state should carry both **conversation messages** (for `ToolNode` / model loops) and **MCP-specific metadata** (prefetched resources, approval flags, execution trace).

In [ ]:
class SupportGraphState(TypedDict):
    messages: Annotated[list, add_messages]
    policy_context: str
    human_approved: bool
    graph_path: list[str]
    status: str


def prefetch_resources_node(state: SupportGraphState) -> dict[str, Any]:
    """Read MCP resources before the model turn — policies belong in context, not tool calls."""
    blocks = []
    for res in mcp_client.resources:
        text = mcp_client.read_resource(res["uri"])
        blocks.append(f"## {res['uri']}\n{text}")
    policy_context = "\n\n".join(blocks)
    system = SystemMessage(content=f"ShopCo support policies (from MCP resources):\n\n{policy_context}")
    return {
        "policy_context": policy_context,
        "messages": [system],
        "graph_path": state.get("graph_path", []) + ["prefetch_resources"],
    }


print("Graph state schema defined.")

---

## 5. Offline Model Node (Planner Stand-In)

When `USE_LIVE_LLM` is false, a deterministic planner mimics tool-calling model behavior — same graph shape, no API cost.

In [ ]:
def offline_planner_node(state: SupportGraphState) -> dict[str, Any]:
    has_tool_results = any(isinstance(m, ToolMessage) for m in state["messages"])
    if has_tool_results:
        snippets = [str(m.content)[:80] for m in state["messages"] if isinstance(m, ToolMessage)]
        answer = "Here is what I found from ShopCo systems: " + " | ".join(snippets)
        return {
            "messages": [AIMessage(content=answer)],
            "status": "completed",
            "graph_path": state["graph_path"] + ["planner:respond"],
        }

    user = next((m.content for m in state["messages"] if isinstance(m, HumanMessage)), "")
    q = user.lower()
    tool_calls: list[dict[str, Any]] = []

    order_match = re.search(r"ORD-[0-9]{4}", user.upper())
    if order_match or "order" in q or "tracking" in q:
        oid = order_match.group(0) if order_match else "ORD-1001"
        tool_calls.append({
            "name": "get_order",
            "args": {"order_id": oid},
            "id": "tc_get_order",
            "type": "tool_call",
        })

    if "refund" in q:
        oid = order_match.group(0) if order_match else "ORD-1001"
        tool_calls.append({
            "name": "create_refund_request",
            "args": {"order_id": oid, "reason": "changed_mind", "amount_usd": 89.50},
            "id": "tc_refund",
            "type": "tool_call",
        })

    if tool_calls:
        return {
            "messages": [AIMessage(content="", tool_calls=tool_calls)],
            "status": "tool_pending",
            "graph_path": state["graph_path"] + ["planner:tool_calls"],
        }

    return {
        "messages": [AIMessage(content="How can I help with your ShopCo order today?")],
        "status": "completed",
        "graph_path": state["graph_path"] + ["planner:chat"],
    }


def live_planner_node(state: SupportGraphState) -> dict[str, Any]:
    from langchain_openai import ChatOpenAI

    llm = ChatOpenAI(model="gpt-4o-mini", api_key=OPENAI_API_KEY, temperature=0)
    bound = llm.bind_tools(LANGCHAIN_TOOLS)
    response = bound.invoke(state["messages"])
    status = "tool_pending" if response.tool_calls else "completed"
    return {
        "messages": [response],
        "status": status,
        "graph_path": state["graph_path"] + [f"planner:live:{status}"],
    }


def planner_node(state: SupportGraphState) -> dict[str, Any]:
    if USE_LIVE_LLM:
        return live_planner_node(state)
    return offline_planner_node(state)

---

## 6. Human Gate Before Destructive MCP Tools

MCP `destructiveHint` annotations signal tools that need approval. In LangGraph, route through a **gate node** before `ToolNode` executes refund requests.

In [ ]:
def human_gate_node(state: SupportGraphState) -> dict[str, Any]:
    last = state["messages"][-1]
    if not isinstance(last, AIMessage) or not last.tool_calls:
        return {"graph_path": state["graph_path"] + ["gate:skip"]}

    destructive = [tc["name"] for tc in last.tool_calls if bridge.is_destructive(tc["name"])]
    if destructive and not state.get("human_approved", False):
        names = ", ".join(destructive)
        return {
            "status": "awaiting_human_approval",
            "messages": [AIMessage(content=f"[HITL pause] Approval required for: {names}. Re-run with human_approved=True.")],
            "graph_path": state["graph_path"] + ["gate:blocked"],
        }
    return {"graph_path": state["graph_path"] + ["gate:approved"]}


def route_after_gate(state: SupportGraphState) -> Literal["tools", "end"]:
    if state.get("status") == "awaiting_human_approval":
        return "end"
    last = state["messages"][-1]
    if isinstance(last, AIMessage) and last.tool_calls:
        return "tools"
    return "end"

---

## 7. Assemble the LangGraph Workflow

Graph topology:

```
START → prefetch_resources → planner → human_gate → tools → planner → END
                              └─ (blocked) ────────────────→ END
```

`ToolNode` runs LangChain tools that delegate to MCP under the hood.

In [ ]:
MCP_TOOL_NODE = ToolNode(LANGCHAIN_TOOLS)

builder = StateGraph(SupportGraphState)
builder.add_node("prefetch_resources", prefetch_resources_node)
builder.add_node("planner", planner_node)
builder.add_node("human_gate", human_gate_node)
builder.add_node("tools", MCP_TOOL_NODE)

builder.add_edge(START, "prefetch_resources")
builder.add_edge("prefetch_resources", "planner")
builder.add_edge("planner", "human_gate")
builder.add_conditional_edges("human_gate", route_after_gate, {"tools": "tools", "end": END})
builder.add_edge("tools", "planner")

checkpointer = MemorySaver()
support_graph = builder.compile(checkpointer=checkpointer)

print("LangGraph compiled with MCP-backed ToolNode.")

---

## 8. Running the Graph — Order Status Ticket

In [ ]:
def run_ticket(question: str, *, human_approved: bool = False, thread_id: str | None = None) -> SupportGraphState:
    config = {"configurable": {"thread_id": thread_id or str(uuid.uuid4())}}
    initial: SupportGraphState = {
        "messages": [HumanMessage(content=question)],
        "policy_context": "",
        "human_approved": human_approved,
        "graph_path": [],
        "status": "running",
    }
    return support_graph.invoke(initial, config=config)


out1 = run_ticket("Where is my order ORD-1001? Need tracking info.")
print("Path:", " → ".join(out1["graph_path"]))
print("Status:", out1["status"])
print("Answer:", out1["messages"][-1].content)

---

## 9. Running the Graph — Refund With Human Gate

In [ ]:
blocked = run_ticket("I want a refund for ORD-1001 please.", human_approved=False)
print("Without approval:")
print("  status:", blocked["status"])
print("  message:", blocked["messages"][-1].content)

approved = run_ticket("I want a refund for ORD-1001 please.", human_approved=True)
print("\nWith approval:")
print("  status:", approved["status"])
print("  message:", approved["messages"][-1].content[:120], "...")
print("  refund tickets:", len(REFUND_LOG))

---

## 10. Checkpointing and Resume

`MemorySaver` stores graph state per `thread_id`. Inspect a paused checkpoint, then re-run the same ticket with `human_approved=True` on the same thread to continue the support conversation.

In [ ]:
thread = "ticket-refund-42"
config = {"configurable": {"thread_id": thread}}

paused = run_ticket("Refund ORD-1002 — changed my mind", human_approved=False, thread_id=thread)
print("First pass status:", paused["status"])

snapshot = support_graph.get_state(config)
print("Checkpoint thread_id:", config["configurable"]["thread_id"])
print("Messages in checkpoint:", len(snapshot.values.get("messages", [])))

# Human approves — re-run ticket on same thread with approval flag
resumed = run_ticket("Refund ORD-1002 — changed my mind", human_approved=True, thread_id=thread)
print("Approved pass status:", resumed["status"])
print("Path tail:", resumed["graph_path"][-5:])

---

## 11. MCP Call Log from the Graph Run

The bridge records every MCP `tools/call` triggered by `ToolNode` — attach this to your observability stack.

In [ ]:
print(f"MCP calls via bridge: {len(bridge.call_log)}")
for entry in bridge.call_log[-5:]:
    flag = "ERR" if entry["is_error"] else "OK"
    print(f"  [{flag}] {entry['tool']} {entry['arguments']}")

---

## 12. Multi-Server Tool Namespacing

When multiple MCP servers expose `search` or `get_status`, prefix tool names at bridge time to avoid collisions in `ToolNode`.

In [ ]:
def namespaced_tool_name(server_alias: str, tool_name: str) -> str:
    return f"{server_alias}__{tool_name}"


def parse_namespaced(name: str) -> tuple[str, str]:
    server, _, tool = name.partition("__")
    return server, tool or name


SERVERS = {
    "shopco": mcp_client.tools,
    "billing": [{"name": "get_invoice", "description": "Billing MCP stub"}],
}

catalog = [
    namespaced_tool_name(server, t["name"])
    for server, tools in SERVERS.items()
    for t in tools
]

print("Namespaced catalog:")
for name in catalog:
    server, tool = parse_namespaced(name)
    print(f"  {name}  (server={server}, tool={tool})")

---

## 13. Resources vs Tools in the Graph

| Data need | Graph pattern | MCP method |
|-----------|---------------|------------|
| Policy text for every ticket | `prefetch_resources` node at graph start | `resources/read` |
| Live order status | `ToolNode` on demand | `tools/call` get_order |
| Refund side effect | `ToolNode` after human gate | `tools/call` create_refund_request |

Prefetching resources into `SystemMessage` keeps the tool menu small — the model sees policies without spending a tool turn.

---

## 14. Production Wiring Checklist

- [ ] MCP client connects at app startup (or lazy per workflow)
- [ ] Tool bridge rebuilds when server emits new `tools/list` version
- [ ] Resource prefetch node runs before first model call
- [ ] `destructiveHint` tools route through HITL gate or `interrupt_before`
- [ ] `thread_id` maps to support ticket ID for checkpoint resume
- [ ] Bridge call log exported to traces (tool name, latency, redacted args)
- [ ] Multi-server tools namespaced before `ToolNode`
- [ ] Transport upgraded from in-process to stdio/SSE for real MCP servers

---

## 15. Common Mistakes

| Mistake | Symptom | Fix |
|---------|---------|-----|
| Hard-code tools, skip discovery | Drift when MCP server updates | Bridge from `tools/list` |
| Put policies in tools | Wasted tool turns | `resources/read` + prefetch node |
| No gate on destructive MCP tools | Unauthorized refunds | `human_gate` + `destructiveHint` |
| Same tool name from 2 servers | Wrong backend called | Namespace `server__tool` |
| Giant graph state | Checkpoint bloat | Store resource URIs, not full blobs |
| MCP errors as plain strings | Model cannot recover | Structured `isError` → ToolMessage text |

---

## 16. Architecture Comparison

| Approach | Pros | Cons |
|----------|------|------|
| Local `@tool` registry | Simple, fast | Tight coupling to code deploys |
| MCP + LangGraph (this notebook) | Discoverable, versioned servers | More moving parts |
| MCP inside model node only | Fewer graph nodes | Harder to test prefetch/HITL separately |
| Remote LangGraph + remote MCP | Full separation | Network latency, auth complexity |

---

## 17. Optional Live LLM Driver

Flip `USE_LIVE_LLM` in the setup cell to `True`, re-run setup through graph compile, then invoke the graph. The model receives the same MCP-backed `LANGCHAIN_TOOLS` discovered at startup.

In [ ]:
if USE_LIVE_LLM:
    live_out = run_ticket("What's the return policy and status of ORD-1001?")
    print(live_out["messages"][-1].content)
else:
    print("Offline mode active. Tools available to graph:", [t.name for t in LANGCHAIN_TOOLS])
    print("Set USE_LIVE_LLM = True in the setup cell to drive the graph with gpt-4o-mini.")

---

## 18. Quiz

1. Why convert MCP tools to LangChain `StructuredTool` objects instead of calling MCP directly inside the model node?
2. Where should policy documents enter the graph — prefetch node or `ToolNode`?
3. How does `destructiveHint` connect to LangGraph human gates?
4. What does `MemorySaver` + `thread_id` enable for blocked refund flows?
5. Why namespace tools as `server__tool_name` when using multiple MCP servers?

---

## 19. Summary

**LangGraph** owns orchestration: prefetch → plan → gate → execute → respond. **MCP** owns capability delivery: discovered tools and resources behind a standard JSON-RPC interface.

The integration hinge is the **MCPToolBridge** — it turns `tools/list` manifests into `ToolNode`-compatible tools that call `tools/call` at runtime. Resources flow through a dedicated **prefetch node** into `SystemMessage` context.

For ShopCo Support Hub, this gives you checkpointed tickets, human approval before destructive refunds, and an observability log of every MCP invocation — the shape you deploy when moving from a local tool registry to protocol-based integrations.